# MAP estimation on a real patient fiber field

Minimal run and analysis. The ground truth is the fiber field of a real patient (`RealDataGroundTruthStrategy`); the activation times are *synthetic*, forward-solved from that field and sampled sparsely with noise, as no measured activation times are available. The prior mean is a constant angle, the one that fits the observations best (scanned below, or `MEAN_CONSTANT`), and the optimization starts at that mean.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv
from ls_bayesian.common.logging import BaseLogger, LoggerSettings

from bayes_cep.mesh.geodesic import assemble_edge_length_graph, compute_graph_distances
from bayes_cep.mesh.interpolation import NearestNeighborInterpolationStrategy
from bayes_cep.mesh.io import load_pyvista_mesh
from bayes_cep.posterior.builder import PosteriorBuilder, PosteriorSettings
from bayes_cep.posterior.eikonal_map import EikonalParameterToSolutionMap
from bayes_cep.posterior.likelihood import LikelihoodSettings
from bayes_cep.posterior.prior import PriorParameters
from bayes_cep.preprocessing.angle_field import build_angle_field_from_fiber_field
from bayes_cep.preprocessing.synthetic_observations import (
    ObservationSamplingSettings,
    generate_synthetic_observations,
)
from bayes_cep.statistics.axial_error_metrics import (
    compute_axial_error_metrics,
    compute_error_by_distance,
    compute_skill_score,
)
from bayes_cep.statistics.axial_statistics import compute_axial_data_diff, wrap_axial_angles
from single_runs.reference import (
    NOISE_VARIANCE,
    NUM_OBSERVATIONS,
    OBSERVATION_SEED,
    reference_eikonal_settings,
    reference_optimizer_strategy,
)

pv.set_jupyter_backend("trame")

## Settings

The patient directory holds `mesh.vtu`, `basis_vecs.npy` and `fiber_field.npy`. Prior parameters, number of observations and noise default to the reference values; the eikonal settings (initial site, velocities) are those of patient 01, so change them for another patient.

In [ ]:
PATIENT_DIR = Path.cwd().parent / "example_data" / "raw"
PRIOR = PriorParameters(kappa=0.5, tau=0.5, seed=0)
OBSERVATIONS = ObservationSamplingSettings(
    num_observations=NUM_OBSERVATIONS, noise_variance=NOISE_VARIANCE, seed=OBSERVATION_SEED
)
# The prior mean, and the start of the optimization: a constant angle [rad]. None takes the constant
# that fits the observations best (scanned in the data section). A constant offset costs the prior
# almost nothing, and the seam of the branch window (-pi/2, pi/2] then lies away from the angles of
# the field, where regions on both sides of it settle on different branches.
MEAN_CONSTANT = None
MAX_NUM_ITERATIONS = None  # None: the reference value (1000); set lower for a quick test

## Data: patient ground truth and synthetic observations

In [ ]:
mesh = load_pyvista_mesh(PATIENT_DIR / "mesh.vtu")
basis_vectors = np.load(PATIENT_DIR / "basis_vecs.npy")
fiber_field = np.load(PATIENT_DIR / "fiber_field.npy")
eikonal_settings = reference_eikonal_settings()
interpolation_strategy = NearestNeighborInterpolationStrategy()
connectivity = mesh.cells.reshape(-1, 4)[:, 1:]
vertex_to_simplex_matrix = interpolation_strategy.assemble_matrix(mesh.points, connectivity)

ground_truth = build_angle_field_from_fiber_field(mesh, fiber_field, basis_vectors)
forward_map = EikonalParameterToSolutionMap(
    mesh, basis_vectors, eikonal_settings, interpolation_strategy
)
observed_vertex_indices, observed_activation_times = generate_synthetic_observations(
    forward_map, ground_truth, OBSERVATIONS
)
if MEAN_CONSTANT is None:
    candidates = np.linspace(-np.pi / 2, np.pi / 2, 25, endpoint=False)
    misfits = [
        np.sum(
            (
                forward_map.evaluate_forward(np.full(mesh.n_points, constant))[observed_vertex_indices]
                - observed_activation_times
            )
            ** 2
        )
        for constant in candidates
    ]
    mean_constant = float(candidates[np.argmin(misfits)])
else:
    mean_constant = MEAN_CONSTANT
prior_mean = np.full(mesh.n_points, mean_constant)
print(f"{mesh.n_points} vertices, {mesh.n_cells} triangles, {observed_vertex_indices.size} observed")
print(f"prior mean and initial guess: constant {mean_constant:.2f} rad")

## Posterior and MAP estimate

In [ ]:
log_posterior = PosteriorBuilder(
    PosteriorSettings(
        mesh=mesh,
        basis_vectors=basis_vectors,
        prior_mean_vector=prior_mean,
        prior_parameters=PRIOR,
        eikonal_settings=eikonal_settings,
        likelihood_settings=LikelihoodSettings(
            num_vertices=mesh.points.shape[0],
            observed_vertex_indices=observed_vertex_indices,
            data_vector=observed_activation_times,
            noise_variance=OBSERVATIONS.noise_variance,
        ),
    ),
    interpolation_strategy=interpolation_strategy,
).build()

strategy = reference_optimizer_strategy()
if MAX_NUM_ITERATIONS is not None:
    strategy.lbfgs_settings.maximum_num_iterations = MAX_NUM_ITERATIONS
with BaseLogger(LoggerSettings(), prefix="map") as logger:
    model, optimizer = strategy.build(log_posterior, logger)
    result = optimizer.run(initial_guess=prior_mean, model=model)
map_estimate = result.result
print(f"success: {result.success}\nstatus: {result.status_message}")

fig, (loss_axis, gradient_axis) = plt.subplots(1, 2, figsize=(10, 4))
loss_axis.semilogy(result.loss_history)
loss_axis.set_xlabel("iteration")
loss_axis.set_ylabel("loss")
gradient_axis.semilogy(result.gradient_norm_history)
gradient_axis.set_xlabel("iteration")
gradient_axis.set_ylabel("gradient norm")
fig.tight_layout()

In [ ]:
np.save("map_patient.npy", map_estimate)

## Estimate vs. ground truth

Compared on simplices, as the forward model sees the field. The error is axial, so a shift by $\pi$ is no error.

In [ ]:
truth_simplex = vertex_to_simplex_matrix @ ground_truth
estimate_simplex = vertex_to_simplex_matrix @ map_estimate
prior_mean_simplex = vertex_to_simplex_matrix @ prior_mean
error = compute_axial_data_diff(estimate_simplex, truth_simplex)
prior_mean_error = compute_axial_data_diff(prior_mean_simplex, truth_simplex)

# The parameter is a real number, but the fiber is axial: plot the angle modulo pi
clim = (-np.pi / 2, np.pi / 2)
for name, values, cmap, limits in (
    ("ground truth (patient)", wrap_axial_angles(truth_simplex), "hsv", clim),
    ("MAP estimate", wrap_axial_angles(estimate_simplex), "hsv", clim),
    ("absolute axial error [deg]", np.degrees(np.abs(error)), "viridis", (0.0, 90.0)),
):
    plot_mesh = mesh.copy()
    plot_mesh.cell_data["value"] = values
    plotter = pv.Plotter()
    plotter.add_mesh(plot_mesh, scalars="value", cmap=cmap, clim=limits)
    plotter.add_text(name)
    plotter.show()

## Error metrics

In [ ]:
metrics = compute_axial_error_metrics(error)
prior_mean_metrics = compute_axial_error_metrics(prior_mean_error)
metrics["skill_score"] = compute_skill_score(
    metrics["error_mean_deg"], prior_mean_metrics["error_mean_deg"]
)
print(f"{'metric':<24}{'prior mean':>12}{'MAP':>12}")
for key, value in metrics.items():
    print(f"{key:<24}{prior_mean_metrics.get(key, float('nan')):>12.4g}{value:>12.4g}")

true_times = forward_map.evaluate_forward(ground_truth)
estimate_times = forward_map.evaluate_forward(map_estimate)
is_observed = np.zeros(mesh.n_points, dtype=bool)
is_observed[observed_vertex_indices] = True
for name, selection in (("all", ~is_observed | is_observed), ("observed", is_observed), ("unobserved", ~is_observed)):
    rmse = np.sqrt(np.mean((estimate_times - true_times)[selection] ** 2))
    print(f"activation-time RMSE, {name:<10}: {rmse:.4f}")

## Error distribution and error vs. distance to the observations

In [ ]:
absolute_error_deg = np.degrees(np.abs(error))
vertex_distances = compute_graph_distances(
    assemble_edge_length_graph(mesh.points, connectivity), observed_vertex_indices, min_over_sources=True
)
quantiles = (0.5, 0.9)
bin_distances, bin_errors = compute_error_by_distance(
    vertex_to_simplex_matrix @ vertex_distances, absolute_error_deg, num_bins=10, quantiles=quantiles
)

fig, (cdf_axis, distance_axis) = plt.subplots(1, 2, figsize=(11, 4))
for name, values in (("prior mean", np.degrees(np.abs(prior_mean_error))), ("MAP estimate", absolute_error_deg)):
    sorted_values = np.sort(values)
    cdf_axis.plot(sorted_values, np.arange(1, sorted_values.size + 1) / sorted_values.size, label=name)
cdf_axis.set_xlabel("absolute axial error [deg]")
cdf_axis.set_ylabel("fraction of simplices below")
cdf_axis.legend()
for quantile, values in zip(quantiles, bin_errors.T, strict=True):
    distance_axis.plot(bin_distances, values, marker="o", label=f"{int(100 * quantile)}th percentile")
distance_axis.set_xlabel("distance to the nearest observation")
distance_axis.set_ylabel("absolute axial error [deg]")
distance_axis.legend()
fig.tight_layout()

## Loss at the MAP estimate vs. at the patient field

The patient field is not a sample of the prior (the $\pi$-branch is resolved globally), so its prior cost can be far above $N/2$. A lower loss at the patient field than at the MAP estimate would mean the optimizer stopped early.

In [ ]:
for name, field in (("MAP estimate", map_estimate), ("patient field", ground_truth)):
    likelihood_cost = log_posterior.evaluate_likelihood_cost(field)
    prior_cost = log_posterior.evaluate_prior_cost(field)
    print(f"{name:<14} loss {likelihood_cost + prior_cost:>12.1f}  (likelihood {likelihood_cost:>10.1f}, prior {prior_cost:>10.1f})")